# `02_xml_conversion.ipynb` — Lossless Canonical JSONL Conversion

## 1. Why Build a Canonical Intermediate Representation?
In the historical 2018 neural baseline (`konvens-2018-german-absa`), `lib/tsv2penn.py` converted the XML into BIO token files (`.cpenn`) with two lossy transformations:
1. When multiple opinions overlapped on the same token span, it collapsed them into a single most-frequent category (`Counter.most_common(1)`).
2. It injected a synthetic dummy token `__D__` at token index 0 to hold implicit `NULL` targets (`from=0, to=0`).

For modern LLMs, both artifacts are harmful. Instead, `xml_parser.py` parses the raw XML directly into typed `CanonicalDocument` records (`data/canonical/*.jsonl` and compact `.jsonl.gz` archives), preserving 100% of XML attributes and opinion multiplicities.

In [ ]:
from pathlib import Path
import os, sys, json
from IPython.display import Image, display, Markdown
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
    os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from germeval_llm.convert import ensure_derived_datasets_ready
ensure_derived_datasets_ready()
print("Repository initialized:", ROOT.name)


In [ ]:
from germeval_llm.xml_parser import load_canonical_jsonl, verify_lossless_roundtrip

manifest = json.loads(Path('data/manifests/dataset_manifest.json').read_text())
for sp, info in manifest['splits'].items():
    docs = load_canonical_jsonl(info['canonical_jsonl'])
    xml_exists = Path(info['source_xml']).exists()
    if xml_exists:
        ok, errs = verify_lossless_roundtrip(info['source_xml'], docs)
        assert ok and not errs
    else:
        ok = info['roundtrip_verified']
    print(f"{sp:10s}: docs={len(docs):6d}, opinions={info['num_opinions']:6d}, roundtrip_ok={ok}, sha256={info['canonical_sha256'][:12]}...")

sample_doc = load_canonical_jsonl('data/canonical/dev.jsonl')[0]
print('\nSample CanonicalDocument (dev[0]):')
print(json.dumps(sample_doc.model_dump(), ensure_ascii=False, indent=2))

### Key Takeaways
- Canonical JSONL preserves all original XML attributes (`from`, `to`, `from2`, `to2`, raw `category`, raw `polarity`) 1-to-1.
- Lossless round-trip verification passes with 0 errors across all 26,209 documents.
- Compact `.jsonl.gz` archives (5 MB total across all 4 splits) are tracked in Git so Google Colab notebooks load all 26,209 canonical documents instantaneously without requiring the raw XML files.